In [6]:
import numpy as np
import pandas as pd
from collections import Counter
import argparse
import time

In [ ]:
DATA_PATH = "NER dataset.csv"
ENCODING = "latin1"
Vocab_size = 200
SUBSET_SIZE = 10000
N_STATES = 5
N_ITERS = 3
VERBOSE = True
RND_SEED = 42

In [ ]:
def load_and_split_sentences(path, encoding="latin1"):
    df = pd.read_csv(path, encoding=encoding, header=0)
    # forward-fill sentence marker and extract numeric id
    df['Sentence #'] = df['Sentence #'].ffill()
    df['sent_id'] = df['Sentence #'].astype(str).str.extract(r'(\d+)', expand=False).astype(int)
    df['Word'] = df['Word'].astype(str)
    sentences = df.groupby('sent_id')['Word'].apply(list).tolist()
    return sentences

def build_vocab(sentences, Vocab_size):
    counter = Counter([w for s in sentences for w in s])
    most_common = [w for w,_ in counter.most_common(Vocab_size)]
    vocab = {w:i for i,w in enumerate(most_common)}
    UNK = "<UNK>"
    vocab[UNK] = len(vocab)
    return vocab, UNK

def sequences_to_ids(sentences, vocab, unk_token):
    def w2i(w): return vocab[w] if w in vocab else vocab[unk_token]
    return [np.array([w2i(w) for w in s], dtype=int) for s in sentences]

def forward_backward_scaled_single(pi, A, B, obs):
    N = A.shape[0]; T = len(obs)
    alpha = np.zeros((T, N)); c = np.zeros(T)
    # forward
    alpha[0] = pi * B[:, obs[0]]
    c[0] = alpha[0].sum()
    if c[0] == 0: c[0] = 1e-300
    alpha[0] /= c[0]
    for t in range(1, T):
        alpha[t] = (alpha[t-1] @ A) * B[:, obs[t]]
        c[t] = alpha[t].sum()
        if c[t] == 0: c[t] = 1e-300
        alpha[t] /= c[t]
    # backward
    beta = np.zeros((T, N))
    beta[-1] = 1.0 / c[-1]
    for t in range(T-2, -1, -1):
        beta[t] = (A @ (B[:, obs[t+1]] * beta[t+1])) / c[t]
    # gamma, xi
    gamma = (alpha * beta)
    gamma /= gamma.sum(axis=1, keepdims=True)
    xi = np.zeros((T-1, N, N))
    for t in range(T-1):
        num = (alpha[t][:,None] * A) * (B[:, obs[t+1]] * beta[t+1])[None,:]
        denom = num.sum()
        if denom == 0: denom = 1e-300
        xi[t] = num / denom
    loglik = np.sum(np.log(c))
    return alpha, beta, gamma, xi, loglik

def baum_welch_sequences(seqs, N_states, M_obs, n_iter=20, tol=1e-4, seed=0, verbose=True, init_params=None):
    rng = np.random.RandomState(seed)
    # init params (allow passing initial pi,A,B via init_params)
    if init_params is None:
        pi = rng.rand(N_states); pi /= pi.sum()
        A = rng.rand(N_states, N_states); A /= A.sum(axis=1, keepdims=True)
        B = rng.rand(N_states, M_obs); B /= B.sum(axis=1, keepdims=True)
    else:
        pi, A, B = init_params
        # if B has wrong shape, reinit B
        if B.shape[1] != M_obs:
            B = rng.rand(N_states, M_obs); B /= B.sum(axis=1, keepdims=True)
    history = []
    for it in range(n_iter):
        t0 = time.time()
        # accumulators
        pi_acc = np.zeros(N_states)
        A_num = np.zeros((N_states, N_states))
        A_den = np.zeros(N_states)
        B_num = np.zeros((N_states, M_obs))
        B_den = np.zeros(N_states)
        total_loglik = 0.0
        # E-step on each sequence
        for obs in seqs:
            if len(obs) == 0: continue
            alpha, beta, gamma, xi, loglik = forward_backward_scaled_single(pi, A, B, obs)
            total_loglik += loglik
            pi_acc += gamma[0]
            A_num += xi.sum(axis=0)
            A_den += gamma[:-1].sum(axis=0)
            for t, o in enumerate(obs):
                B_num[:, o] += gamma[t]
            B_den += gamma.sum(axis=0)
        # M-step
        pi = pi_acc / pi_acc.sum()
        A = np.divide(A_num, A_den[:,None], where=(A_den[:,None]!=0))
        A = np.where(A.sum(axis=1, keepdims=True)==0, np.ones_like(A)/N_states, A)
        A = A / A.sum(axis=1, keepdims=True)
        B = np.divide(B_num, B_den[:,None], where=(B_den[:,None]!=0))
        B = np.where(B.sum(axis=1, keepdims=True)==0, np.ones_like(B)/M_obs, B)
        B = B / B.sum(axis=1, keepdims=True)
        history.append(total_loglik)
        if verbose:
            print(f"Iter {it:2d}  Log-likelihood = {total_loglik:.6f}  time={(time.time()-t0):.1f}s")
        if it>0 and abs(history[-1]-history[-2]) < tol:
            if verbose: print("Converged (change < tol).")
            break
    return pi, A, B, history

In [ ]:
def main():
    # optionally override constants from CLI
    sentences = load_and_split_sentences(DATA_PATH, encoding=ENCODING)
    print(f"Loaded {len(sentences)} sentences.")
    # optionally subset
    if SUBSET_SIZE is not None and SUBSET_SIZE < len(sentences):
        sentences = sentences[:SUBSET_SIZE]
        print(f"Using subset of {len(sentences)} sentences (SUBSET_SIZE={SUBSET_SIZE})")
    vocab, UNK = build_vocab(sentences, Vocab_size)
    seqs = sequences_to_ids(sentences, vocab, UNK)
    M = len(vocab)
    print(f"Vocab size (Vocab_size + UNK) = {M}")
    print("Starting Baum-Welch training ...")
    pi, A, B, history = baum_welch_sequences(seqs, N_STATES, M, n_iter=N_ITERS, seed=RND_SEED, verbose=VERBOSE)
    print("Training finished.\n")

    # -------------------------
    # PRINT learned parameters
    # -------------------------
    np.set_printoptions(precision=6, suppress=True, linewidth=200)
    N = pi.shape[0]
    print("=== Learned parameters ===")
    print(f"π (initial distribution)  shape = {pi.shape}")
    for i, val in enumerate(pi):
        print(f"pi[{i}] = {val:.6f}")
    print("\nA (transition matrix p)  shape = {}".format(A.shape))
    # print A with rows
    for i in range(N):
        row_str = "  ".join([f"{A[i,j]:.6f}" for j in range(N)])
        print(f"A[{i},:] = [ {row_str} ]")

    # Emission matrix B (e)
    print("\nB (emission matrix e)  shape = {}".format(B.shape))
    TOP_PRINT = 20  # top-K words to print per state if vocab is large
    id_to_word = {idx: w for w, idx in vocab.items()}

    if M <= 50:
        # print full matrix (columns are word ids)
        header = "     " + "  ".join([f"{i:>6}" for i in range(M)])
        print(header)
        for i in range(N):
            row = "  ".join([f"{B[i,j]:.6f}" for j in range(M)])
            print(f"state[{i}]  {row}")
    else:
        # print top words per state (readable)
        print(f"(vocab large: showing top {TOP_PRINT} emitted words per state)")
        for s in range(N):
            top_ids = np.argsort(-B[s])[:TOP_PRINT]
            print(f"\nState {s} top {TOP_PRINT} words (word:prob):")
            for wid in top_ids:
                word = id_to_word.get(wid, "<UNK>")
                print(f"  {word} : {B[s,wid]:.6f}")
    print("\nLog-likelihood history (last values):", history[-10:])

if __name__ == "__main__":
    main()

Loaded 47959 sentences.
Using subset of 10000 sentences (SUBSET_SIZE=10000)
Vocab size (Vocab_size + UNK) = 201
Starting Baum-Welch training ...
Iter  0  Log-likelihood = -1179500.909188  time=2.7s
Iter  1  Log-likelihood = -669364.222710  time=2.7s
Iter  2  Log-likelihood = -667792.408206  time=2.6s
Training finished.

=== Learned parameters ===
π (initial distribution)  shape = (5,)
pi[0] = 0.271068
pi[1] = 0.405894
pi[2] = 0.125634
pi[3] = 0.195705
pi[4] = 0.001700

A (transition matrix p)  shape = (5, 5)
A[0,:] = [ 0.083586  0.043670  0.398060  0.179574  0.295110 ]
A[1,:] = [ 0.008789  0.568358  0.311394  0.050497  0.060962 ]
A[2,:] = [ 0.128162  0.286284  0.303848  0.152059  0.129647 ]
A[3,:] = [ 0.409052  0.122802  0.166099  0.123818  0.178230 ]
A[4,:] = [ 0.441749  0.137727  0.252941  0.155921  0.011661 ]

B (emission matrix e)  shape = (5, 201)
(vocab large: showing top 20 emitted words per state)

State 0 top 20 words (word:prob):
  <UNK> : 0.648434
  the : 0.034618
  in : 0.0